# Chadstone Parking Occupancy Analysis

This notebook analyses the historical occupancy of the Chadstone Shopping Centre car parks.
The data is scraped on a regular schedule and published to `data/parking.csv` in this repository.

Each row records how many spaces are **occupied** and **vacant** in a given car park at a point in time.
The `total_occupied` and `total_vacant` columns show the site-wide totals for that timestamp.

We use [Polars](https://pola.rs) for data handling and the lightweight [`xy`](https://github.com/factorish/xy) library for charts.

> **Note** — `xy` charts do not render inline in VSCode. Every chart is saved to
> a self-contained HTML file in the [`charts/`](../charts) folder. Open those in a browser.

## 1. Setup

Import the libraries and prepare the output directory.

In [1]:
from pathlib import Path

import polars as pl
import xy

# All generated charts land in the charts/ folder at the repo root.
CHARTS = Path("../charts")
CHARTS.mkdir(parents=True, exist_ok=True)

## 2. Load the data

Read the latest parking history directly from the GitHub-hosted CSV, then parse
`retrieved_at` into a proper datetime column.

In [2]:
df = pl.read_csv(
    "https://github.com/jay-stein/chaddy-parking-scraper/blob/master/data/parking.csv?raw=true"
).with_columns(
    pl.col("retrieved_at").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

df.shape

(5925, 6)

## 3. Explore the data

Get a feel for the dataset: available car parks and the time range covered.

In [3]:
df.select("car_park").unique().sort("car_park")

car_park
str
"""A"""
"""B"""
"""C"""
"""E"""
"""F"""


In [4]:
df.select(
    pl.col("retrieved_at").min().alias("earliest"),
    pl.col("retrieved_at").max().alias("latest"),
)

earliest,latest
datetime[μs],datetime[μs]
2026-07-12 14:44:24,2026-08-01 14:43:39


## 4. Car Park B occupancy over time

Filter down to Car Park B, sort by time, then draw a line chart.
The chart is saved as a self-contained HTML file in
[`charts/carpark_b.html`](../charts/carpark_b.html).

In [5]:
subset_df = df.filter(pl.col("car_park") == "B").sort("retrieved_at")
subset_df

retrieved_at,car_park,occupied,vacant,total_occupied,total_vacant
datetime[μs],str,i64,i64,i64,i64
2026-07-12 14:44:24,"""B""",986,94,7745,1499
2026-07-12 15:13:59,"""B""",994,86,7791,1453
2026-07-12 18:01:35,"""B""",972,108,5374,3870
2026-07-12 19:18:45,"""B""",599,481,2523,6721
2026-07-12 19:22:38,"""B""",571,509,2365,6879
…,…,…,…,…,…
2026-08-01 13:00:39,"""B""",985,95,6740,2504
2026-08-01 13:30:32,"""B""",992,88,7097,2147
2026-08-01 14:00:44,"""B""",983,97,7215,2029


In [6]:
chart = xy.line_chart(
    xy.line(
        subset_df["retrieved_at"],
        subset_df["occupied"],
        color="#7c3aed",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Occupied Spaces"),
    xy.tooltip(
        title="{x:%d %b %Y %H:%M}",
        format={"y": ",.0f"},
    ),
    title="Occupied Spaces in Car Park B Over Time",
)

CHARTS.joinpath("carpark_b.html").write_text(chart.to_html(), encoding="utf-8")
print("Saved -> charts/carpark_b.html")

Saved -> charts/carpark_b.html


## 5. Total occupancy across all car parks

Sum the occupied spaces over every car park per timestamp to see how busy the
whole centre is at each sample.

Saved to [`charts/total_occupancy.html`](../charts/total_occupancy.html).

In [7]:
total_df = (
    df.group_by("retrieved_at")
    .agg(pl.sum("occupied").alias("total_occupied"))
    .sort("retrieved_at")
)
total_df

retrieved_at,total_occupied
datetime[μs],i64
2026-07-12 14:44:24,7745
2026-07-12 15:13:59,7791
2026-07-12 18:01:35,5374
2026-07-12 19:18:45,2523
2026-07-12 19:22:38,2365
…,…
2026-08-01 13:00:39,6740
2026-08-01 13:30:32,7097
2026-08-01 14:00:44,7215


In [8]:
total_chart = xy.line_chart(
    xy.line(
        total_df["retrieved_at"],
        total_df["total_occupied"],
        color="#0ea5e9",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Total Occupied Spaces"),
    xy.tooltip(
        title="{x:%d %b %Y %H:%M}",
        format={"y": ",.0f"},
    ),
    title="Total Occupancy Across All Car Parks",
)

CHARTS.joinpath("total_occupancy.html").write_text(total_chart.to_html(), encoding="utf-8")
print("Saved -> charts/total_occupancy.html")

Saved -> charts/total_occupancy.html
